[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataguirre/Curso-IA-Aplicada/blob/main/Semana%2014_RAG/implementacion_RAG.ipynb)

# Implementacion de un RAG con abstracts del repositorio de Banrep

In [ ]:
!pip install -U --quiet "transformers>=4.44.0" "accelerate>=0.33.0" "bitsandbytes>=0.43.1" "peft>=0.12.0" sentencepiece
!pip install -U --quiet --no-cache-dir bitsandbytes

In [ ]:
import os, sys, time
print("Reiniciando el runtime para activar bitsandbytes…")
time.sleep(1)
os.kill(os.getpid(), 9)

In [ ]:
!pip -q install langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers ragas unidecode

In [ ]:
import pandas as pd

banrep = pd.read_csv('https://drive.google.com/uc?id=1vz3530nztdYPs-x0ZELlRPjNEcK453hc')
columns = ['uuid', 'name', 'collection', 'authors', 'type', 'date', 'abstract_spa',
           'doi_link', 'access_rights', 'access_rights', 'keywords_spa', 'jel_spa']

banrep = banrep[(~banrep['abstract_spa'].isna()) & (~banrep['authors'].isna())][columns].reset_index(drop=True)
banrep

In [ ]:
import re

def clean_text(s:str) -> str:
    s = str(s) if s is not None else ""
    s = s.replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

banrep = banrep.copy()
banrep['name'] = banrep['name'].fillna("").map(clean_text)
banrep['abstract_spa'] = banrep['abstract_spa'].fillna("").map(clean_text)

# Filtra filas sin contenido y crea el texto base
banrep = banrep[banrep['abstract_spa'] != ""]
banrep['text'] = (banrep['name'] + ". " + banrep['abstract_spa']).str.strip()

len(banrep), banrep[['name','abstract_spa']].head(2)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

docs_texts, docs_meta = [], []
for row in banrep.itertuples():
    chunks = splitter.split_text(row.text)
    for i, ch in enumerate(chunks):
        docs_texts.append(ch)
        docs_meta.append({
            "title": row.name,
            "chunk": i
        })

len(docs_texts)

## Modelo de *Embeddings*: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

Este modelo es un **encoder multilingüe** entrenado por *Sentence-Transformers* que convierte textos en vectores numéricos (embeddings) que capturan su significado semántico.  
Es decir, dos frases con significados similares quedarán representadas por vectores cercanos en el espacio vectorial.

### Características principales
- **Arquitectura base:** MiniLM (una versión ligera de BERT).  
- **Idiomas soportados:** más de 50 idiomas, incluido el español.  
- **Tamaño reducido:** 33 millones de parámetros → ideal para correr rápido en CPU o GPU T4.  
- **Uso típico:** búsqueda semántica, clustering, y recuperación de documentos (retrieval).

### Rol en el pipeline RAG
Dentro de nuestro sistema RAG (Retrieval-Augmented Generation), este modelo se usa en la etapa de **retrieval** para:
1. Convertir cada *abstract* del Banco de la República en un vector (embedding).  
2. Convertir la pregunta del usuario en otro vector.  
3. Calcular la similitud entre ambos mediante la **distancia coseno**.  
4. Recuperar los documentos más semánticamente cercanos a la consulta.

Así, el RAG no busca por palabras exactas como un motor de texto clásico, sino por **significado**.  
Esto permite preguntas naturales como:

> “¿Qué efectos tuvo la inflación en la balanza de pagos?”  

y el sistema podrá encontrar abstracts que mencionen *“efectos de los precios internacionales sobre el comercio exterior”*, aunque no usen exactamente la palabra *inflación*.

En resumen, este modelo nos da la base para **recuperar información relevante por contenido y contexto**, no solo por coincidencia de palabras.

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

EMB_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMB_MODEL)

vectorstore = FAISS.from_texts(texts=docs_texts, embedding=embeddings, metadatas=docs_meta)
# Opcional: persistir
vectorstore.save_local("/content/banrep_faiss_index")

In [ ]:
USE_RERANK = True

if USE_RERANK:
    RERANK_MODEL = "mixedbread-ai/mxbai-rerank-base-v1"
    rr_tok = AutoTokenizer.from_pretrained(RERANK_MODEL)
    rr_model = AutoModelForSequenceClassification.from_pretrained(RERANK_MODEL)
    rr_model = rr_model.to("cuda" if torch.cuda.is_available() else "cpu").eval()

    def rerank(query, candidates, top_n=5):
        pairs = [(query, c.page_content) for c in candidates]
        enc = rr_tok([p[0] for p in pairs], [p[1] for p in pairs],
                     truncation=True, padding=True, return_tensors="pt")
        enc = {k:v.to(rr_model.device) for k,v in enc.items()}
        with torch.no_grad():
            scores = rr_model(**enc).logits.squeeze(-1).float().detach().cpu().numpy()
        order = np.argsort(-scores)
        return [candidates[i] for i in order[:top_n]]

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

tok = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL, device_map="auto", quantization_config=bnb_cfg
)

## Funciones principales del RAG

A continuación se describen las funciones que componen el flujo del sistema RAG (Retrieval-Augmented Generation).  
Cada una cumple un papel específico dentro del proceso de **recuperación, preparación y generación de respuestas.**

---

### `retrieve(query, k=10, search_type="similarity")`

Esta función realiza la **búsqueda semántica**.  
- Usa el índice FAISS (`vectorstore`) para encontrar los *k* documentos más similares a la consulta (`query`), según los embeddings creados previamente.  
- El parámetro `search_type` puede ser `"similarity"` o `"mmr"` (Maximal Marginal Relevance, que diversifica resultados).  
- Devuelve una lista de documentos relevantes con su texto y metadatos.

*Ejemplo:* si el usuario pregunta sobre “inflación y tasas de interés”, esta función recupera abstracts que hablan de esos temas aunque no usen exactamente las mismas palabras.

---

### `build_prompt(query, contexts)`

Esta función construye el **prompt** que se enviará al modelo generador (LLM).  
- Crea una instrucción inicial que le dice al modelo responder **solo con la información del contexto**.  
- Combina todos los fragmentos (`contexts`) recuperados y los etiqueta con un índice `[1], [2], [3]…` junto con su título.  
- Añade la pregunta del usuario al final.

*Propósito:* asegurar que el modelo tenga toda la información relevante y sepa cómo citarla.

---

### `generate_answer(prompt, max_new_tokens=400, temperature=0.2)`

Esta función se encarga de la **generación del texto final**.  
- Envía el *prompt* al modelo de lenguaje (`llm`) para que produzca una respuesta.  
- El parámetro `max_new_tokens` limita la longitud de la respuesta.  
- `temperature` controla la creatividad (valores bajos → respuestas más precisas y determinísticas).  
- Devuelve el texto generado en formato legible.

---

### `ask(query, k=10, rerank_top=5, mmr=False, show_sources=True)`

Es la **función principal del RAG**, que integra todos los pasos anteriores.

1. Llama a `retrieve()` para buscar los documentos más relevantes.  
2. Si está activo `USE_RERANK`, reordena los resultados con el modelo de re-ranking.  
3. Construye el *prompt* con `build_prompt()`.  
4. Genera la respuesta final con `generate_answer()`.  
5. (Opcional) Muestra las fuentes utilizadas al final del texto.

In [ ]:
from textwrap import shorten

def retrieve(query, k=10, search_type="similarity"):
    retriever = vectorstore.as_retriever(search_type=search_type, search_kwargs={"k": k})
    return retriever.get_relevant_documents(query)

def build_prompt(query, contexts):
    header = (
        "Responde en español usando EXCLUSIVAMENTE el contexto.\n"
        "Si la respuesta no aparece en el contexto, di explícitamente que no está disponible.\n\n"
    )
    ctx = ""
    for i, d in enumerate(contexts):
        t = d.metadata.get("title", "")
        ctx += f"[{i+1}] {d.page_content}\n(Cita: {t})\n\n"
    user = f"Pregunta: {query}\n\nCita las fuentes como [n] cuando corresponda."
    return header + "Contexto:\n" + ctx + user

def generate_answer(prompt, max_new_tokens=400, temperature=0.2):
    input_ids = tok(prompt, return_tensors="pt").to(llm.device)
    out = llm.generate(**input_ids, max_new_tokens=max_new_tokens,
                       temperature=temperature, do_sample=False)
    return tok.decode(out[0], skip_special_tokens=True)

def ask(query, k=10, rerank_top=5, mmr=False, show_sources=True):
    hits = retrieve(query, k=k, search_type=("mmr" if mmr else "similarity"))
    if not hits:
        return "No encontré contexto relevante."

    if USE_RERANK:
        hits = rerank(query, hits, top_n=min(rerank_top, len(hits)))
    else:
        hits = hits[:min(rerank_top, len(hits))]

    prompt = build_prompt(query, hits)
    ans = generate_answer(prompt)

    if show_sources:
        sources = []
        for i, h in enumerate(hits):
            title = shorten(h.metadata.get("title",""), 90, placeholder="…")
            sources.append(f"[{i+1}] {title}")
        ans += "\n\nFuentes:\n" + "\n".join(sources)
    return ans

In [ ]:
print(ask("¿Qué hallazgos mencionan los abstracts sobre inflación y expectativas entre 2019 y 2023?"))

In [ ]:
print(ask("¿Cómo relacionan los documentos las tasas de interés con el crédito en Colombia?"))

In [ ]:
print(ask("¿Qué se reporta sobre el mercado laboral en los informes recientes?"))

# Actividad

## 1. Experimenta con los parámetros del RAG

En esta sección probaremos cómo cambian las respuestas del modelo al modificar los parámetros del RAG.

- **`k`** controla cuántos documentos se recuperan inicialmente del índice FAISS.  
- **`rerank_top`** define cuántos de esos documentos conserva el re-ranker.  
- **`USE_RERANK`** activa o desactiva el modelo de reordenamiento.

Prueba diferentes combinaciones y observa:
1. Qué tan completa o precisa es la respuesta.  
2. Si el modelo alucina o responde con datos no presentes en el contexto.  
3. Cuánto tarda en generar la respuesta.

Podrias anotar tus observaciones en una tabla como esta:

| k | rerank_top | Re-rank | Calidad percibida | Tiempo (s) |
|--:|--:|:--:|:--:|--:|
| 5 | 3 | ❌ | Parcial | 3.2 |
| 10 | 5 | ✅ | Completa | 5.8 |


## 2. Explora los *chunks* recuperados

Antes de generar una respuesta, es importante entender **qué textos está usando el modelo**.

En esta parte imprime los *chunks* (fragmentos) que el retriever considera más relevantes.  
Esto te permitirá observar:
- Si realmente son pertinentes a la pregunta.  
- Si el tamaño de los *chunks* o el solapamiento podrían mejorarse.  

Modifica los valores de `chunk_size` y `chunk_overlap` en el *splitter* y repite la búsqueda para comparar resultados.

## 3. Evalúa el desempeño del RAG

Aquí evaluaremos la calidad de las respuestas generadas por el sistema.

Puedes usar dos enfoques:
1. **Evaluación manual:** comparar la respuesta del modelo con un texto de referencia (*gold*).  
2. **Evaluación automática:** usar métricas como *faithfulness* o *answer relevancy* (por ejemplo, con la librería `ragas`).

Diseña 3–5 preguntas con respuestas esperadas y mide:
- Qué tan fiel es la respuesta al contexto.  
- Qué tan relevante es respecto a la pregunta original.


## 4. Mejora tu RAG

En esta parte aplica tu creatividad para mejorar el pipeline.

Algunas ideas:
- Implementar un **retriever híbrido** (BM25 + embeddings).  
- Probar otro modelo de **embeddings** o **LLM generador**.  
- Ajustar el *chunking* para optimizar precisión y costo.  
- Agregar un campo de **citas automáticas** con los títulos recuperados.

El objetivo es comparar tu versión mejorada con el baseline inicial:  
¿es más precisa?, ¿más rápida?, ¿más confiable?